In [1]:
# ==========================================================
# RAG WITH CHATGPT + HISTORY
#  INITIALIZATION
# ==========================================================

import os
from dotenv import load_dotenv

from pinecone import Pinecone
from sentence_transformers import SentenceTransformer

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage


# ==========================================================
# LOAD ENVIRONMENT VARIABLES
# ==========================================================

load_dotenv()

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
INDEX_NAME = os.getenv("PINECONE_INDEX_NAME2")


# ==========================================================
# CONFIGURATION
# ==========================================================

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

CHAT_MODEL = "gpt-3.5-turbo"

TOP_K = 8

MIN_SCORE = 0.35

FALLBACK_MIN_SCORE = 0.25


# ==========================================================
# DOCUMENT CATEGORIES
# ==========================================================

CATEGORY_SOURCES = {

    "crop_advisory": [

        "ICAR_Kharif_Agro_Advisories_2025",
        "ICAR_Maize_Package_of_Practices",
        "ICAR_Wheat_Cultivation_Pocket_Guide",

    ],

    "soil_health": [

        "ICAR_Soil_Nutrient_Management",
        "SoilHealthCard_Implementation_Guidelines",

    ],

    "pest_control": [

        "NIPHM_IPM_Package_Rice",

    ],

    "scheme": [

        "PMFBY_Operational_Guidelines",
        "PM_KMY_Operational_Guidelines",
        "Kisan_Credit_Card_Guidelines",

    ]

}


# ==========================================================
# ROUTING KEYWORDS
# ==========================================================

PEST_KEYWORDS = [

    "pest",
    "insect",
    "disease",
    "fungus",
    "borer",
    "spray"

]

SOIL_KEYWORDS = [

    "soil",
    "soil health",
    "soil health card",
    "soil testing",
    "soil nutrient"

]

SCHEME_KEYWORDS = [

    "pmfby",
    "pm-kmy",
    "pm kmy",
    "pm-kisan",
    "kisan credit card",
    "kcc",
    "crop insurance",
    "fasal bima",
    "pension"

]


# ==========================================================
# FILE NAME SUFFIXES
# ==========================================================

SOURCE_SUFFIXES = [

    "",
    ".pdf",
    ".txt",
    ".doc",
    ".docx"

]


# ==========================================================
# SYSTEM PROMPT
# ==========================================================

SYSTEM_PROMPT = """
You are an Agricultural AI Assistant for Indian farmers.

Rules:

1. Answer ONLY from the retrieved CONTEXT.

2. Never use outside knowledge.

3. If the answer is not available in the context, reply exactly:

I don't have verified information on this in my knowledge base.

4. If the user asks a follow-up question like:
   - explain this
   - tell me more
   - what are those
   - explain them
   - its benefits

use the conversation history to understand what "this", "those", or "it" refers to.

5. Keep answers practical, simple and farmer-friendly.
"""


# ==========================================================
# CONNECT TO PINECONE
# ==========================================================

print("Connecting to Pinecone...")

pc = Pinecone(api_key=PINECONE_API_KEY)

index = pc.Index(INDEX_NAME)

print("Connected Successfully")

print("Index :", INDEX_NAME)


# ==========================================================
# LOAD EMBEDDING MODEL
# ==========================================================

print("\nLoading Embedding Model...")

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

print("Embedding Model Loaded")


# ==========================================================
# LOAD CHAT MODEL
# ==========================================================

print("\nLoading ChatGPT Model...")

llm = ChatOpenAI(

    api_key=OPENAI_API_KEY,

    model=CHAT_MODEL,

    temperature=0.2

)

print("ChatGPT Model Loaded")


# ==========================================================
# MEMORY
# ==========================================================

chat_history = []

last_category = None


print("\nInitialization Completed Successfully.")

Connecting to Pinecone...


PineconeValueError: Either name or host must be provided to create an Index client.

In [ ]:
# ==========================================================
# FUNCTIONS
# EXPAND SOURCE NAMES
# ==========================================================

def expand_source_names(source_names):

    expanded = []

    for source in source_names:

        expanded.append(source)

        for suffix in SOURCE_SUFFIXES:

            expanded.append(source + suffix)

    return list(set(expanded))


# ==========================================================
# CATEGORY ROUTER
# ==========================================================

def route_category(query):

    global last_category

    query = query.lower()

    if any(word in query for word in PEST_KEYWORDS):

        last_category = "pest_control"

        return last_category

    if any(word in query for word in SOIL_KEYWORDS):

        last_category = "soil_health"

        return last_category

    if any(word in query for word in SCHEME_KEYWORDS):

        last_category = "scheme"

        return last_category

    return last_category


# ==========================================================
# RETRIEVE DOCUMENTS
# ==========================================================

def retrieve(query, category=None):

    query_vector = embedding_model.encode(query).tolist()

    search_args = {

        "vector": query_vector,

        "top_k": TOP_K,

        "include_metadata": True

    }

    if category and category in CATEGORY_SOURCES:

        search_args["filter"] = {

            "source": {

                "$in": expand_source_names(

                    CATEGORY_SOURCES[category]

                )

            }

        }

    try:

        results = index.query(**search_args)

    except Exception:

        search_args.pop("filter", None)

        results = index.query(**search_args)

    matches = [

        match

        for match in results.matches

        if match.score >= MIN_SCORE

    ]

    if len(matches) == 0:

        search_args.pop("filter", None)

        results = index.query(**search_args)

        matches = [

            match

            for match in results.matches

            if match.score >= FALLBACK_MIN_SCORE

        ]

    if len(matches) == 0:

        matches = results.matches[:TOP_K]

    return matches


# ==========================================================
# QUERY REWRITING
# ==========================================================

def rewrite_query(history, question):

    if len(history) == 0:

        return question

    conversation = ""

    for item in history[-6:]:

        conversation += f"{item['role']} : {item['content']}\n"

    prompt = f"""

You are helping an Agricultural AI Assistant.

Using the conversation history below,
rewrite ONLY the latest user question into
a complete standalone search query.

Conversation:

{conversation}

Latest Question:

{question}

Rules:

1. Resolve words like:
   - it
   - this
   - those
   - they
   - them

2. Preserve the original meaning.

3. Return ONLY the rewritten question.

"""

    response = llm.invoke([

        HumanMessage(content=prompt)

    ])

    rewritten = response.content.strip()

    if rewritten == "":

        return question

    return rewritten


# ==========================================================
# GENERATE FINAL ANSWER
# ==========================================================

def generate_answer(question, history, matches):

    if len(matches) == 0:

        return "I don't have verified information on this in my knowledge base."

    context = "\n\n----------------------\n\n".join(

        f"Source : {m.metadata.get('source','Unknown')}\n\n"

        f"{m.metadata.get('text','')}"

        for m in matches

    )

    conversation = ""

    for item in history[-6:]:

        conversation += f"{item['role']} : {item['content']}\n"

    prompt = f"""

Conversation History

{conversation}

Context

{context}

Question

{question}

Answer ONLY using the Context.

If the answer is not found in the context,
reply exactly:

I don't have verified information on this in my knowledge base.

"""

    response = llm.invoke([

        SystemMessage(content=SYSTEM_PROMPT),

        HumanMessage(content=prompt)

    ])

    return response.content.strip()


# ==========================================================
# COMPLETE RAG PIPELINE
# ==========================================================

def answer_question(history, question):

    standalone_query = rewrite_query(

        history,

        question

    )

    category = route_category(

        standalone_query

    )

    matches = retrieve(

        standalone_query,

        category

    )

    answer = generate_answer(

        question,

        history,

        matches

    )

    sources = sorted(

        {

            m.metadata.get(

                "source",

                "Unknown"

            )

            for m in matches

        }

    )

    confidence = max(

        (

            m.score

            for m in matches

        ),

        default=0

    )

    return {

        "question": question,

        "rewritten_query": standalone_query,

        "category": category,

        "answer": answer,

        "sources": sources,

        "confidence_score": confidence

    }


print("All Functions Loaded Successfully.")

In [ ]:
# ==========================================================
# CHAT LOOP
# ==========================================================

print("=" * 70)
print("🌾 Agricultural AI Assistant")
print("Type 'exit', 'quit' or 'stop' to end the conversation.")
print("=" * 70)

while True:

    try:

        question = input("\nAsk your question : ").strip()

    except KeyboardInterrupt:

        print("\n\nSession Ended.")
        break

    except EOFError:

        print("\n\nSession Ended.")
        break


    # ======================================================
    # EXIT
    # ======================================================

    if question.lower() in ["exit", "quit", "stop"]:

        print("\nThank you for using Agricultural AI Assistant.")
        print("Session Ended.")

        break


    # ======================================================
    # EMPTY QUESTION
    # ======================================================

    if question == "":

        print("Please enter a question.")

        continue


    # ======================================================
    # GET ANSWER
    # ======================================================

    try:

        result = answer_question(

            chat_history,

            question

        )

    except Exception as e:

        print("\nError :", e)

        continue


    # ======================================================
    # DISPLAY OUTPUT
    # ======================================================

    print("\n" + "=" * 70)

    print("Question")
    print("-" * 70)
    print(result["question"])


    if result["rewritten_query"].strip().lower() != result["question"].strip().lower():

        print("\nRewritten Query")
        print("-" * 70)
        print(result["rewritten_query"])


    print("\nDetected Category")
    print("-" * 70)

    if result["category"]:

        print(result["category"])

    else:

        print("General Search")


    print("\nAnswer")
    print("-" * 70)
    print(result["answer"])


    print("\nSources")
    print("-" * 70)

    if result["sources"]:

        for source in result["sources"]:

            print("•", source)

    else:

        print("No source found")


    if result["confidence_score"] > 0:

        print("\nTop Retrieval Score :", round(result["confidence_score"], 4))

        print(

            "Similarity Score :",

            f"{result['confidence_score']*100:.2f}%"

        )

    else:

        print("\nTop Retrieval Score : N/A")


    # ======================================================
    # SAVE CHAT HISTORY
    # ======================================================

    chat_history.append(

        {

            "role": "User",

            "content": question

        }

    )

    chat_history.append(

        {

            "role": "Assistant",

            "content": result["answer"]

        }

    )


    # ======================================================
    # LIMIT HISTORY SIZE
    # ======================================================

    if len(chat_history) > 20:

        chat_history = chat_history[-20:]